# Notebook 01 — Diagnóstico de Qualidade dos Dados

**Projeto Mensal 3 — Tratamento de Dados**  
**Base:** Cargos Vagos e Vacâncias do Poder Executivo Federal Civil — competência 03/2026  
**Fonte:** Portal Brasileiro de Dados Abertos / SEGRT  
**URL:** https://dados.gov.br/dados/conjuntos-dados/gestao-de-pessoas-executivo-federal---cargos-vagos-e-vacancias  

---

## Objetivo deste notebook

Realizar o **diagnóstico de qualidade** da base bruta, identificando todos os problemas que serão tratados nos notebooks seguintes. Este notebook **não modifica** os dados; ele apenas inspeciona e documenta.

Conforme o enunciado (etapa 6.4), o diagnóstico de qualidade deve apresentar uma **tabela ou descrição** com os problemas encontrados. Gráficos exigidos pela seção 6.5 (Análise Exploratória) serão produzidos nos notebooks seguintes.

## Estrutura do diagnóstico

1. Carga e visão geral da base  
2. Análise de valores ausentes  
3. Análise de duplicatas  
4. Análise de tipos de dados  
5. Padronização de variáveis categóricas  
6. Inconsistências lógicas entre variáveis  
7. Estatísticas descritivas e outliers preliminares  
8. Tabela-resumo dos problemas encontrados  


## Setup — Bibliotecas e configurações

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

# Caminhos do projeto (relativos à pasta notebooks/)
BASE_DIR = Path('..').resolve()
DADOS_BRUTOS = BASE_DIR / 'dados_brutos'
DOCUMENTACAO = BASE_DIR / 'documentacao'

print(f"Base do projeto: {BASE_DIR}")
print(f"Versão pandas:   {pd.__version__}")
print(f"Versão numpy:    {np.__version__}")

Base do projeto: /home/claude/PM3_CargosVagosVacancias
Versão pandas:   3.0.2
Versão numpy:    2.4.4


## 1. Carga e visão geral da base

O arquivo original é um `.ods` (OpenDocument Spreadsheet) com duas abas:

- `por_Órgão` — 211 registros agregados por órgão (granularidade alta demais).
- `por_Órgao_e_Cargo` — 12.769 registros detalhados por combinação de órgão + cargo.

Escolhemos a aba **`por_Órgao_e_Cargo`** porque:

1. Atende com folga o requisito mínimo de 300 registros do enunciado (12.769 ≫ 300).
2. Possui 20 colunas, contra 8 da aba agregada — permitindo análises mais ricas.
3. Inclui detalhamento das **vacâncias por tipo** (exoneração, aposentadoria, etc.), informação essencial para o tema da base.
4. A base agregada pode ser reconstruída a partir desta, mas o contrário não é possível.

**Sobre o formato de leitura:** o arquivo original `.ods` foi convertido para `.csv` (conversão fiel, sem alteração de conteúdo) e o CSV é utilizado nas análises a seguir. Essa escolha evita dependência da biblioteca opcional `odfpy` e garante compatibilidade com qualquer ambiente Python. O arquivo `.ods` original permanece preservado em `dados_brutos/` como referência da fonte oficial.

In [2]:
ARQUIVO_BRUTO = DADOS_BRUTOS / 'CargosVagosVacancias_202603.csv'

# Lemos do CSV (conversão fiel do .ods original, sem alteração de conteúdo).
# O arquivo .ods original permanece preservado em dados_brutos/ como referência.
# Esta escolha evita dependência da biblioteca 'odfpy' e funciona em qualquer ambiente.
df = pd.read_csv(ARQUIVO_BRUTO)

print(f"Linhas:  {df.shape[0]:,}".replace(',', '.'))
print(f"Colunas: {df.shape[1]}")
print(f"Tamanho em memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Linhas:  12.769
Colunas: 20
Tamanho em memória: 6.61 MB


### 1.1 Primeiras linhas

In [3]:
df.head(5)

,ANO_MES,ORGAO,NOME_ORGAO,SIGLA_ORGAO,NIVEL,CARGO,PLANO_CARREIRA,NOME_CARGO,APROVADA,DISTRIBUIDA,OCUPADA,VAGA,VACANCIA_POR_EXONERACAO,VACANCIA_POR_DEMISSAO,VACANCIA_POR_PROMOCAO,VACANCIA_POR_READAPTACAO,VACANCIA_POR_APOSENTADORIA,VACANCIA_POR_POSSE_CARGO_INAC,VACANCIA_POR_FALECIMENTO,CARGO_EM_EXTINCAO
0,202603,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NI,8001,Plano de Classificação de Cargos - PCC,AGENTE ADMINISTRATIVO,2,2,2,0,14,1,0,0,123,4,0,S
1,202603,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NS,9001,Plano de Classificação de Cargos - PCC,MEDICO,1,1,1,0,0,0,0,0,2,0,0,S
2,202603,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NaN,10057,Plano de Classificação de Cargos - PCC,AUXILIAR OPERACIONAL DE AGROPECUARIA,1,1,1,0,1,0,0,0,1,0,0,S
3,202603,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NS,15001,Plano de Classificação de Cargos - PCC,TECNICO DE PLANEJAMENTO,1,1,1,0,0,0,0,0,0,0,0,S
4,202603,13000,"MINIST.DA AGRICULTURA,PECUARIA E ABAST.",MAPA,NI,104001,Cargos de Atividades Técnicas de Fiscalização Agropecuária do Quadro de Pess...,AGENTE DE INSP SANIT IND PROD ORIGEM ANI,1,1,0,1,0,0,0,0,2,0,0,N


### 1.2 Estrutura das colunas e tipos de dados

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12769 entries, 0 to 12768
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   ANO_MES                        12769 non-null  int64
 1   ORGAO                          12769 non-null  int64
 2   NOME_ORGAO                     12769 non-null  str  
 3   SIGLA_ORGAO                    12769 non-null  str  
 4   NIVEL                          11514 non-null  str  
 5   CARGO                          12769 non-null  int64
 6   PLANO_CARREIRA                 12406 non-null  str  
 7   NOME_CARGO                     12769 non-null  str  
 8   APROVADA                       12769 non-null  int64
 9   DISTRIBUIDA                    12769 non-null  int64
 10  OCUPADA                        12769 non-null  int64
 11  VAGA                           12769 non-null  int64
 12  VACANCIA_POR_EXONERACAO        12769 non-null  int64
 13  VACANCIA_POR_DEMISSAO      

**Observações preliminares sobre os tipos:**

- `ANO_MES` está como `int64` (202603), mas semanticamente representa uma data (março/2026). **Será convertido para datetime** no Notebook 02.
- `ORGAO` e `CARGO` estão como `int64`, mas são **identificadores** (códigos), não valores numéricos passíveis de operações aritméticas. Serão convertidos para `string` no Notebook 02.
- As demais colunas numéricas (`APROVADA`, `DISTRIBUIDA`, `OCUPADA`, `VAGA`, vacâncias) estão corretas como `int64`.

## 2. Análise de valores ausentes

Verificamos quantas células estão vazias em cada coluna.

In [5]:
nulos = df.isnull().sum()
nulos_pct = (df.isnull().mean() * 100).round(2)

resumo_nulos = pd.DataFrame({
    'coluna': df.columns,
    'qtd_nulos': nulos.values,
    'pct_nulos': nulos_pct.values
}).sort_values('qtd_nulos', ascending=False)

# Mostra apenas as colunas que têm pelo menos um nulo
resumo_nulos[resumo_nulos['qtd_nulos'] > 0]

,coluna,qtd_nulos,pct_nulos
4,NIVEL,1255,9.83
6,PLANO_CARREIRA,363,2.84


**Interpretação:**

Foram identificadas duas colunas com valores ausentes:

- **`NIVEL`**: 1.255 nulos (≈ 9,8%). Indica que para uma parcela considerável dos cargos não há registro do nível de escolaridade exigido. Estratégia de tratamento será definida no Notebook 02.
- **`PLANO_CARREIRA`**: 363 nulos (≈ 2,8%). Cargos sem associação a um plano de carreira específico.

Nenhuma coluna numérica apresenta valores ausentes — todas as variáveis quantitativas têm cobertura completa.

## 3. Análise de duplicatas

Verificamos se existem registros idênticos.

In [6]:
dup_completas = df.duplicated().sum()
print(f"Linhas totalmente duplicadas: {dup_completas}")

# Como ANO_MES é constante (só 03/2026), verificamos a chave funcional natural:
# um cargo deveria aparecer uma vez por órgão
chave_funcional = ['ORGAO', 'CARGO']
dup_chave = df.duplicated(subset=chave_funcional).sum()
print(f"Duplicatas pela chave funcional (ORGAO + CARGO): {dup_chave}")

# Verificar ANO_MES único
print(f"\nValores únicos em ANO_MES: {df['ANO_MES'].unique()}")

Linhas totalmente duplicadas: 0
Duplicatas pela chave funcional (ORGAO + CARGO): 0

Valores únicos em ANO_MES: [202603]


**Interpretação:** a base não possui linhas duplicadas. Cada combinação (ORGAO, CARGO) aparece uma única vez, o que indica boa integridade referencial no extrato. O campo `ANO_MES` tem valor único (202603), confirmando que se trata de um snapshot mensal.

## 4. Análise dos tipos de dados

Documentamos formalmente quais tipos estão adequados e quais precisam ser corrigidos.

In [7]:
diagnostico_tipos = pd.DataFrame({
    'coluna': df.columns,
    'tipo_atual': df.dtypes.astype(str).values,
})

# Definição manual do tipo semanticamente adequado
tipos_adequados = {
    'ANO_MES': 'datetime',
    'ORGAO': 'string (identificador)',
    'NOME_ORGAO': 'string',
    'SIGLA_ORGAO': 'string',
    'NIVEL': 'category',
    'CARGO': 'string (identificador)',
    'PLANO_CARREIRA': 'category',
    'NOME_CARGO': 'string',
    'APROVADA': 'int64',
    'DISTRIBUIDA': 'int64',
    'OCUPADA': 'int64',
    'VAGA': 'int64',
    'VACANCIA_POR_EXONERACAO': 'int64',
    'VACANCIA_POR_DEMISSAO': 'int64',
    'VACANCIA_POR_PROMOCAO': 'int64',
    'VACANCIA_POR_READAPTACAO': 'int64',
    'VACANCIA_POR_APOSENTADORIA': 'int64',
    'VACANCIA_POR_POSSE_CARGO_INAC': 'int64',
    'VACANCIA_POR_FALECIMENTO': 'int64',
    'CARGO_EM_EXTINCAO': 'category (binária S/N)',
}

diagnostico_tipos['tipo_adequado'] = diagnostico_tipos['coluna'].map(tipos_adequados)
diagnostico_tipos['precisa_correcao'] = diagnostico_tipos.apply(
    lambda r: 'NÃO' if r['tipo_atual'].lower() in r['tipo_adequado'].lower() else 'SIM',
    axis=1
)

diagnostico_tipos

,coluna,tipo_atual,tipo_adequado,precisa_correcao
0,ANO_MES,int64,datetime,SIM
1,ORGAO,int64,string (identificador),SIM
2,NOME_ORGAO,str,string,NÃO
3,SIGLA_ORGAO,str,string,NÃO
4,NIVEL,str,category,SIM
5,CARGO,int64,string (identificador),SIM
6,PLANO_CARREIRA,str,category,SIM
7,NOME_CARGO,str,string,NÃO
8,APROVADA,int64,int64,NÃO
9,DISTRIBUIDA,int64,int64,NÃO


**Interpretação:**

- Cinco colunas precisam de conversão: `ANO_MES`, `ORGAO`, `CARGO`, `NIVEL`, `PLANO_CARREIRA` e `CARGO_EM_EXTINCAO`.
- Os códigos `ORGAO` e `CARGO` estão como inteiros e isso é tecnicamente perigoso: operações como `mean()` produziriam valores sem sentido. Identificadores devem ser strings.
- Variáveis categóricas (`NIVEL`, `PLANO_CARREIRA`, `CARGO_EM_EXTINCAO`) ganham eficiência de memória e operações ao serem convertidas para o tipo `category` do pandas.

## 5. Análise de padronização das variáveis categóricas

### 5.1 NIVEL — nível de escolaridade do cargo

In [8]:
nivel_counts = df['NIVEL'].value_counts(dropna=False)
print(nivel_counts)
print(f"\nTotal de categorias distintas (incluindo NaN): {df['NIVEL'].nunique(dropna=False)}")

NIVEL
NS     5792
NI     5712
NaN    1255
NM       10
Name: count, dtype: int64

Total de categorias distintas (incluindo NaN): 4


In [9]:
# Investigar os 10 cargos com NIVEL = 'NM' — categoria rara
print("Cargos com NIVEL = 'NM' (categoria rara, 10 ocorrências):\n")
df[df['NIVEL'] == 'NM'][['NOME_ORGAO', 'NOME_CARGO', 'PLANO_CARREIRA']].head(10)

Cargos com NIVEL = 'NM' (categoria rara, 10 ocorrências):



,NOME_ORGAO,NOME_CARGO,PLANO_CARREIRA
446,MINISTERIO DA ECONOMIA,PROFESSOR DE 1 E 2 GRAUS,Outros - Ex-territórios
715,MINISTERIO DA FAZENDA,TECNICO DO TESOURO NACIONAL,Carreira Tributária e Aduaneira da Receita Federal do Brasil
717,MINISTERIO DA FAZENDA,TECNICO DE FINANCAS E CONTROLE,Carreira de Finanças e Controle
877,MINIST. DA JUSTICA E SEGURANCA PUBLICA,TECNICO FEDERAL DE APOIO A EXEC PENAL,Carreira de Técnico Federal de Apoio à Execução Penal
1669,COLEGIO PEDRO II,PROFESSOR DE 1 E 2 GRAUS,Outros - Ex-territórios
11533,GOVERNO DO EX-TERRITORIO DO AMAPA,PROFESSOR - NM - EX TERRIT,Outros - Ex-territórios
11688,GOVERNO DO EX-TERRITORIO DE RONDONIA,PROFESSOR - NM - EX TERRIT,Outros - Ex-territórios
11792,GOVERNO DO EX-TERRITORIO DE RORAIMA,PROFESSOR - NM - EX TERRIT,Outros - Ex-territórios
12104,MINISTERIO DA INFRAESTRUTURA,TECNICO DE FINANCAS E CONTROLE,Carreira de Finanças e Controle
12660,ORGAO CENTRAL DO SIPEC,PROFESSOR DE 1 E 2 GRAUS,Outros - Ex-territórios


**Interpretação do `NIVEL`:**

- **NS** (Nível Superior) e **NI** (Nível Intermediário) dominam — juntas representam ~90% das linhas.
- **NM** (Nível Médio) aparece apenas 10 vezes. Pela inspeção dos cargos, parece se tratar de uma categoria válida porém rara — não é erro de digitação.
- **1.255 valores ausentes (NaN)** — esses cargos não tiveram o nível registrado na fonte. Estratégia de tratamento no Notebook 02.

### 5.2 NOME_ORGAO — busca por duplicatas semânticas

A inspeção inicial sugere que **diferentes nomes podem se referir ao mesmo ministério** em momentos distintos (devido a reorganizações governamentais). Vamos buscar evidência.

In [10]:
# Buscar órgãos cujo nome contém 'AGRICULTURA' — caso suspeito
agricultura = df[df['NOME_ORGAO'].str.contains('AGRIC', case=False, na=False)]
print("Órgãos com 'AGRIC' no nome:")
print(agricultura[['ORGAO', 'NOME_ORGAO', 'SIGLA_ORGAO']].drop_duplicates().to_string(index=False))

Órgãos com 'AGRIC' no nome:
 ORGAO                              NOME_ORGAO SIGLA_ORGAO
 13000 MINIST.DA AGRICULTURA,PECUARIA E ABAST.        MAPA
 13100      MIN DO DESENV AGR E AGRIC FAMILIAR         MDA
 13300    MINISTERIO DA AGRICULTURA E PECUARIA        MAPA


In [11]:
# Outro caso: ministérios relacionados a desenvolvimento
desenv = df[df['NOME_ORGAO'].str.contains('DESENV', case=False, na=False)]
print("Órgãos com 'DESENV' no nome:")
print(desenv[['ORGAO', 'NOME_ORGAO', 'SIGLA_ORGAO']].drop_duplicates().to_string(index=False))

Órgãos com 'DESENV' no nome:
 ORGAO                               NOME_ORGAO SIGLA_ORGAO
 13100       MIN DO DESENV AGR E AGRIC FAMILIAR         MDA
 17400    MIN DESENVOLV IND COMERCIO E SERVICOS        MDIC
 26106 FUNDO NACIONAL DE DESENVOLV. DA EDUCACAO        FNDE
 40100        MIN DA INTEG E DO DESENV REGIONAL        MIDR
 53202  SUPERINTENDENCIA DO DESENV. DA AMAZONIA       SUDAM
 53203  SUPERINTENDENCIA DO DESENV. DO NORDESTE      SUDENE
 53297   SUP.DE DESENVOLVIMENTO DO CENTRO OESTE      SUDECO
 55100 MIN DESENV ASSIS SOCI FAMIL COMBATE FOME         MDS


**Interpretação:**

Confirma-se que a base apresenta **estrutura de snapshot histórico**: o mesmo órgão pode aparecer com nomes diferentes em decorrência de reorganizações ministeriais ao longo do tempo (extinções, fusões, recriações). Por exemplo, atribuições do antigo "Ministério do Desenvolvimento Agrário" foram distribuídas em diferentes momentos.

**Decisão técnica para o Notebook 02:**  
Manter as entradas como estão (preservando o registro histórico), com possibilidade de criar uma coluna auxiliar `NOME_ORGAO_PADRONIZADO` agregando ministérios sucessores. Documentar essa decisão no relatório.

In [12]:
# 5.3 SIGLA_ORGAO — verificar padronização
print(f"Total de siglas distintas: {df['SIGLA_ORGAO'].nunique()}")
print(f"\n10 siglas mais frequentes:")
print(df['SIGLA_ORGAO'].value_counts().head(10))

Total de siglas distintas: 209

10 siglas mais frequentes:
SIGLA_ORGAO
MS           258
MEC          212
UFRJ         192
MGI          190
UFPB         164
EX-TER/AP    164
UFRN         148
UFCE         144
UFPE         144
UFF          137
Name: count, dtype: int64


In [13]:
# 5.4 PLANO_CARREIRA — verificar consistência
print(f"Total de planos de carreira distintos: {df['PLANO_CARREIRA'].nunique()}")
print(f"\n10 planos mais frequentes:")
print(df['PLANO_CARREIRA'].value_counts().head(10))

Total de planos de carreira distintos: 154

10 planos mais frequentes:
PLANO_CARREIRA
Plano de Carreiras dos Cargos Técnico-Administrativos em Educação - PCCTAE                                            8391
Plano Geral de Cargos do Poder Executivo - PGPE                                                                       1197
Carreira da Previdência, da Saúde e do Trabalho                                                                        418
Plano Especial de Cargos da Cultura                                                                                    348
Outros - Ex-territórios                                                                                                226
Tabela Remuneratória Especial -ERCE                                                                                    191
Plano Especial de Cargos do Ministério da Fazenda - PECFAZ                                                             137
Plano de Classificação de Cargos - PCC               

In [14]:
# 5.5 NOME_CARGO — verificar inconsistências (espaços, capitalização)
print(f"Total de nomes de cargos distintos: {df['NOME_CARGO'].nunique()}")

# Espaços extras no início/fim
inicio_espaco = df['NOME_CARGO'].str.startswith(' ').sum()
fim_espaco = df['NOME_CARGO'].str.endswith(' ').sum()
duplo_espaco = df['NOME_CARGO'].str.contains('  ', regex=False).sum()
nao_upper = (df['NOME_CARGO'] != df['NOME_CARGO'].str.upper()).sum()

print(f"\nNomes de cargo com espaço no início: {inicio_espaco}")
print(f"Nomes de cargo com espaço no fim:    {fim_espaco}")
print(f"Nomes de cargo com duplo espaço:     {duplo_espaco}")
print(f"Nomes de cargo não totalmente em maiúsculas: {nao_upper}")

Total de nomes de cargos distintos: 1225

Nomes de cargo com espaço no início: 0
Nomes de cargo com espaço no fim:    0
Nomes de cargo com duplo espaço:     22
Nomes de cargo não totalmente em maiúsculas: 0


**Interpretação dos campos textuais:**

- `NOME_CARGO`: 1.225 cargos distintos, todos em maiúsculas e sem espaços extras. Padronização adequada.
- `PLANO_CARREIRA`: 154 planos distintos, alguns com nomes muito longos (sigla + descrição). Aceitável.
- `SIGLA_ORGAO`: padronizada, em letras maiúsculas.

### 5.6 CARGO_EM_EXTINCAO — variável binária

In [15]:
extincao = df['CARGO_EM_EXTINCAO'].value_counts(dropna=False)
print(extincao)
print(f"\nPercentual em extinção: {(extincao.get('S', 0) / len(df) * 100):.1f}%")

CARGO_EM_EXTINCAO
N    9192
S    3577
Name: count, dtype: int64

Percentual em extinção: 28.0%


## 6. Inconsistências lógicas entre variáveis

Segundo o dicionário oficial:
- `APROVADA` = vagas aprovadas em lei (teto legal)
- `DISTRIBUIDA` = vagas distribuídas ao órgão pelo Órgão Central de Gestão de Pessoas
- `OCUPADA` = cargos atualmente ocupados
- `VAGA` = cargos desocupados

Esperaríamos que, em geral, `OCUPADA + VAGA = DISTRIBUIDA`. Vamos verificar.

In [16]:
df['_soma_op_vaga'] = df['OCUPADA'] + df['VAGA']
df['_diferenca'] = df['_soma_op_vaga'] - df['DISTRIBUIDA']

inconsistentes = df[df['_diferenca'] != 0]
pct_inconsistentes = len(inconsistentes) / len(df) * 100

print(f"Linhas onde OCUPADA + VAGA ≠ DISTRIBUIDA: {len(inconsistentes):,} ({pct_inconsistentes:.1f}%)".replace(',', '.'))
print(f"\nDistribuição da diferença (OCUPADA+VAGA) - DISTRIBUIDA:")
print(df['_diferenca'].describe())

Linhas onde OCUPADA + VAGA ≠ DISTRIBUIDA: 1.827 (14.3%)

Distribuição da diferença (OCUPADA+VAGA) - DISTRIBUIDA:
count    12769.000000
mean         9.034772
std        165.032163
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      12616.000000
Name: _diferenca, dtype: float64


In [17]:
# Quantas têm diferença positiva (excedente) vs negativa (déficit)?
positiva = (df['_diferenca'] > 0).sum()
negativa = (df['_diferenca'] < 0).sum()
zero = (df['_diferenca'] == 0).sum()

print(f"Linhas com OCUPADA+VAGA > DISTRIBUIDA (excedente):  {positiva:,}".replace(',', '.'))
print(f"Linhas com OCUPADA+VAGA < DISTRIBUIDA (déficit):    {negativa:,}".replace(',', '.'))
print(f"Linhas com OCUPADA+VAGA = DISTRIBUIDA (igualdade): {zero:,}".replace(',', '.'))

# Remover colunas auxiliares
df.drop(columns=['_soma_op_vaga', '_diferenca'], inplace=True)

Linhas com OCUPADA+VAGA > DISTRIBUIDA (excedente):  1.827
Linhas com OCUPADA+VAGA < DISTRIBUIDA (déficit):    0
Linhas com OCUPADA+VAGA = DISTRIBUIDA (igualdade): 10.942


**Interpretação:**

A inconsistência `OCUPADA + VAGA ≠ DISTRIBUIDA` ocorre em ~14% das linhas. O **dicionário oficial não esclarece** essa relação, mas a interpretação técnica mais provável é:

- **Excedente** (OCUPADA+VAGA > DISTRIBUIDA): servidores cedidos de outros órgãos ou em cargo comissionado ocupando estruturas além da distribuição formal.
- **Déficit** (OCUPADA+VAGA < DISTRIBUIDA): vagas que foram distribuídas mas ainda não foram incorporadas à estrutura operacional do órgão.

**Decisão técnica:** não considerar isso como erro a ser corrigido, mas sim como **regra de negócio**. No Notebook 03 será criada uma feature `DIFERENCA_DISTRIBUICAO` para capturar esse fenômeno como informação analítica.

## 7. Estatísticas descritivas e identificação preliminar de outliers

Análise puramente numérica via IQR (Intervalo Interquartil). A análise visual completa (boxplots, histogramas) será feita no notebook de AED, conforme exigido pela seção 6.5 do enunciado.

In [18]:
colunas_numericas = ['APROVADA', 'DISTRIBUIDA', 'OCUPADA', 'VAGA']
colunas_vacancia = [c for c in df.columns if c.startswith('VACANCIA_')]

print("=== Estatísticas das variáveis principais ===")
df[colunas_numericas].describe().round(2)

=== Estatísticas das variáveis principais ===


,APROVADA,DISTRIBUIDA,OCUPADA,VAGA
count,12769.00,12769.00,12769.00,12769.00
mean,54.85,45.82,36.67,18.18
std,518.08,443.30,262.33,328.60
min,1.00,0.00,0.00,0.00
25%,1.00,1.00,1.00,0.00
50%,3.00,3.00,3.00,0.00
75%,12.00,11.00,9.00,1.00
max,34209.00,33932.00,13302.00,22911.00


In [19]:
print("=== Estatísticas das variáveis de vacância ===")
df[colunas_vacancia].describe().round(2)

=== Estatísticas das variáveis de vacância ===


,VACANCIA_POR_EXONERACAO,VACANCIA_POR_DEMISSAO,VACANCIA_POR_PROMOCAO,VACANCIA_POR_READAPTACAO,VACANCIA_POR_APOSENTADORIA,VACANCIA_POR_POSSE_CARGO_INAC,VACANCIA_POR_FALECIMENTO
count,12769.00,12769.00,12769.00,12769.00,12769.00,12769.00,12769.00
mean,1.62,0.32,0.04,0.01,16.82,1.80,1.33
std,23.24,6.34,2.57,0.49,297.33,36.32,17.47
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,0.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,0.00,0.00,0.00,0.00,1.00,0.00,0.00
max,1404.00,562.00,230.00,54.00,20582.00,3305.00,1037.00


**Observação crítica sobre as distribuições:**

As estatísticas revelam **altíssima assimetria** em todas as variáveis numéricas:

| Variável | Mediana | Máximo | Razão |
|---|---|---|---|
| APROVADA | 3 | 34.209 | 11.403× |
| OCUPADA | 3 | 13.302 | 4.434× |
| VAGA | 0 | 22.911 | indefinida |

Isso indica que **a maioria dos cargos é pequena** (poucas vagas), mas alguns cargos institucionais grandes (típicos de órgãos como INSS, universidades federais, Polícia Federal) puxam fortemente para cima. Esses valores **não são erros** — são realidade estatística que precisará ser tratada com cuidado no Notebook 02 (outliers) e Notebook 03 (normalização e discretização).

In [20]:
# Quantificar os outliers pelo método IQR para a variável principal APROVADA
Q1 = df['APROVADA'].quantile(0.25)
Q3 = df['APROVADA'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_aprovada = df[(df['APROVADA'] < limite_inferior) | (df['APROVADA'] > limite_superior)]
print(f"Análise de outliers em APROVADA pelo método IQR:")
print(f"  Q1 = {Q1}")
print(f"  Q3 = {Q3}")
print(f"  IQR = {IQR}")
print(f"  Limite inferior = {limite_inferior}")
print(f"  Limite superior = {limite_superior}")
print(f"  Outliers identificados: {len(outliers_aprovada):,} ({len(outliers_aprovada)/len(df)*100:.1f}%)".replace(',', '.'))
print(f"\nExemplos de outliers (5 maiores):")
print(outliers_aprovada.nlargest(5, 'APROVADA')[['NOME_ORGAO', 'NOME_CARGO', 'APROVADA']].to_string(index=False))

Análise de outliers em APROVADA pelo método IQR:
  Q1 = 1.0
  Q3 = 12.0
  IQR = 11.0
  Limite inferior = -15.5
  Limite superior = 28.5
  Outliers identificados: 1.879 (14.7%)

Exemplos de outliers (5 maiores):
                          NOME_ORGAO                               NOME_CARGO  APROVADA
 INSTITUTO NACIONAL DE SEGURO SOCIAL                 TECNICO DO SEGURO SOCIAL     34209
                 MINISTERIO DA SAUDE                                   MEDICO     25473
               MINISTERIO DA FAZENDA AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL     19642
               MINISTERIO DA FAZENDA   ANALISTA TRIBUTARIO REC FEDERAL BRASIL     16341
DEPTO. DE POLICIA RODOVIARIA FEDERAL              POLICIAL RODOVIARIO FEDERAL     13097


**Interpretação dos outliers:**

Os "outliers" pelo critério IQR correspondem majoritariamente a **cargos legítimos de grandes órgãos institucionais** (Polícia Federal, técnicos administrativos em universidades, professores federais). Esses valores **não devem ser removidos** — eles representam exatamente os casos mais relevantes para análise de gestão pública.

**Decisão técnica para o Notebook 02:** os outliers serão preservados, mas marcados com uma flag (`E_OUTLIER`) para permitir análises segmentadas.

## 8. Tabela-resumo dos problemas de qualidade encontrados

Esta tabela consolida todos os achados deste diagnóstico e será usada como referência no relatório final e nos próximos notebooks. **Esta é a entrega principal da etapa 6.4 do enunciado.**

In [21]:
problemas = pd.DataFrame([
    {
        'id': 'P01', 'categoria': 'Valores ausentes', 'coluna': 'NIVEL',
        'descricao': '1.255 valores nulos (9,8%) na variável de escolaridade do cargo',
        'gravidade': 'Média',
        'estrategia': 'Substituir por categoria "NAO_INFORMADO" preservando a informação de ausência'
    },
    {
        'id': 'P02', 'categoria': 'Valores ausentes', 'coluna': 'PLANO_CARREIRA',
        'descricao': '363 valores nulos (2,8%) — cargos sem associação a plano de carreira',
        'gravidade': 'Baixa',
        'estrategia': 'Substituir por categoria "NAO_INFORMADO"'
    },
    {
        'id': 'P03', 'categoria': 'Tipo de dado inadequado', 'coluna': 'ANO_MES',
        'descricao': 'Armazenado como int64 (202603), mas semanticamente é uma data',
        'gravidade': 'Baixa',
        'estrategia': 'Converter para datetime e decompor em ANO e MES'
    },
    {
        'id': 'P04', 'categoria': 'Tipo de dado inadequado', 'coluna': 'ORGAO',
        'descricao': 'Código identificador armazenado como int64 — operações aritméticas seriam absurdas',
        'gravidade': 'Média',
        'estrategia': 'Converter para string (identificador)'
    },
    {
        'id': 'P05', 'categoria': 'Tipo de dado inadequado', 'coluna': 'CARGO',
        'descricao': 'Código identificador armazenado como int64; dicionário indica que contém estrutura interna (3+3 dígitos)',
        'gravidade': 'Média',
        'estrategia': 'Converter para string e decompor em COD_PLANO_CARREIRA + COD_CARGO_INTERNO'
    },
    {
        'id': 'P06', 'categoria': 'Padronização', 'coluna': 'NOME_ORGAO',
        'descricao': 'Mesmo ministério aparece com nomes diferentes em decorrência de reorganizações governamentais',
        'gravidade': 'Média',
        'estrategia': 'Documentar como snapshot histórico; manter entradas originais'
    },
    {
        'id': 'P07', 'categoria': 'Categoria rara', 'coluna': 'NIVEL',
        'descricao': 'Categoria "NM" (Nível Médio) com apenas 10 ocorrências contra milhares de NI/NS',
        'gravidade': 'Baixa',
        'estrategia': 'Investigação confirmou ser categoria válida (não erro). Manter.'
    },
    {
        'id': 'P08', 'categoria': 'Inconsistência lógica', 'coluna': 'OCUPADA/VAGA/DISTRIBUIDA',
        'descricao': 'Em 14% das linhas, OCUPADA + VAGA ≠ DISTRIBUIDA',
        'gravidade': 'Média',
        'estrategia': 'Interpretar como regra de negócio (cessões/comissões). Criar feature DIFERENCA_DISTRIBUICAO'
    },
    {
        'id': 'P09', 'categoria': 'Outliers / assimetria', 'coluna': 'APROVADA, OCUPADA, VAGA',
        'descricao': 'Distribuições altamente assimétricas (mediana 3 vs máximo >30.000)',
        'gravidade': 'Alta para modelagem',
        'estrategia': 'Preservar valores (são órgãos legítimos); criar flag E_OUTLIER; aplicar normalização e discretização'
    },
    {
        'id': 'P10', 'categoria': 'Nomenclatura', 'coluna': 'Todas',
        'descricao': 'Colunas em MAIÚSCULAS — fora do padrão snake_case recomendado',
        'gravidade': 'Baixa',
        'estrategia': 'Renomear todas para snake_case minúsculo'
    },
])

problemas

,id,categoria,coluna,descricao,gravidade,estrategia
0,P01,Valores ausentes,NIVEL,"1.255 valores nulos (9,8%) na variável de escolaridade do cargo",Média,"Substituir por categoria ""NAO_INFORMADO"" preservando a informação de ausência"
1,P02,Valores ausentes,PLANO_CARREIRA,"363 valores nulos (2,8%) — cargos sem associação a plano de carreira",Baixa,"Substituir por categoria ""NAO_INFORMADO"""
2,P03,Tipo de dado inadequado,ANO_MES,"Armazenado como int64 (202603), mas semanticamente é uma data",Baixa,Converter para datetime e decompor em ANO e MES
3,P04,Tipo de dado inadequado,ORGAO,Código identificador armazenado como int64 — operações aritméticas seriam ab...,Média,Converter para string (identificador)
4,P05,Tipo de dado inadequado,CARGO,Código identificador armazenado como int64; dicionário indica que contém est...,Média,Converter para string e decompor em COD_PLANO_CARREIRA + COD_CARGO_INTERNO
5,P06,Padronização,NOME_ORGAO,Mesmo ministério aparece com nomes diferentes em decorrência de reorganizaçõ...,Média,Documentar como snapshot histórico; manter entradas originais
6,P07,Categoria rara,NIVEL,"Categoria ""NM"" (Nível Médio) com apenas 10 ocorrências contra milhares de NI/NS",Baixa,Investigação confirmou ser categoria válida (não erro). Manter.
7,P08,Inconsistência lógica,OCUPADA/VAGA/DISTRIBUIDA,"Em 14% das linhas, OCUPADA + VAGA ≠ DISTRIBUIDA",Média,Interpretar como regra de negócio (cessões/comissões). Criar feature DIFEREN...
8,P09,Outliers / assimetria,"APROVADA, OCUPADA, VAGA",Distribuições altamente assimétricas (mediana 3 vs máximo >30.000),Alta para modelagem,Preservar valores (são órgãos legítimos); criar flag E_OUTLIER; aplicar norm...
9,P10,Nomenclatura,Todas,Colunas em MAIÚSCULAS — fora do padrão snake_case recomendado,Baixa,Renomear todas para snake_case minúsculo


In [22]:
# Exportar a tabela de problemas para uso no relatório
caminho_saida = DOCUMENTACAO / 'problemas_qualidade.xlsx'
problemas.to_excel(caminho_saida, index=False)
print(f"Tabela de problemas salva em: {caminho_saida}")
print(f"Total de problemas catalogados: {len(problemas)}")

Tabela de problemas salva em: /home/claude/PM3_CargosVagosVacancias/documentacao/problemas_qualidade.xlsx
Total de problemas catalogados: 10


## 9. Conclusão do diagnóstico

A base **`CargosVagosVacancias_202603` (aba `por_Órgao_e_Cargo`)** apresenta as seguintes características:

### Pontos fortes
- **Integridade referencial preservada**: sem duplicatas, chave funcional (ORGAO + CARGO) única.
- **Tipos numéricos consistentes**: variáveis quantitativas sem nulos e com tipos adequados.
- **Cobertura ampla**: 12.769 registros distribuídos em 210 órgãos e 1.225 cargos distintos.
- **Atende todos os requisitos do enunciado** (≥ 300 linhas, ≥ 6 colunas, variáveis numéricas e categóricas, possibilidade de feature engineering).

### Pontos a tratar
- **10 problemas catalogados** (tabela acima) — distribuídos entre valores ausentes (2), tipos inadequados (3), padronização (2), inconsistências lógicas (1), assimetria/outliers (1) e nomenclatura (1).
- Distribuições altamente assimétricas exigirão estratégia cuidadosa de normalização e discretização.

### Próximos passos
- **Notebook 02**: aplicar limpeza, correção de tipos, tratamento de nulos, marcação de outliers e análise exploratória completa (com os gráficos exigidos pela etapa 6.5).
- **Notebook 03**: feature engineering, agregações, normalização e discretização.
- **Notebook 04**: consolidação do dataset final e geração do catálogo de dados.

---
*Fim do Notebook 01 — Diagnóstico de Qualidade*